In [ ]:
import nltk
nltk.download('punkt', quiet=False)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

nltk utils


In [ ]:
import numpy as np
import random
import json


In [ ]:
from nltk.stem.porter  import PorterStemmer
stemmer = PorterStemmer()

In [ ]:
def tokenize(sentence):
    return nltk.word_tokenize(sentence,language="english")


def stem(word):
    return stemmer.stem(word.lower())



def bag_of_words(tokenize_sentence,all_words):
    sentence_words = [stem(word) for word in tokenize_sentence]
    bag = np.zeros(len(all_words),dtype=np.float32)
    for idx,w in enumerate(all_words):
        if w in sentence_words:
            bag[idx] = 1
    return bag



# Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [ ]:
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)

        return out

# Train

In [ ]:
with open('intents.json', 'r') as f:
    intents = json.load(f)

In [ ]:
all_words = []
tags = []
xy = []

In [ ]:
# loop through each sentence in our intents patterns
for intent in intents['intents']:
    tag = intent['tag']
    # add to tag list
    tags.append(tag)
    for pattern in intent['patterns']:
        # tokenize each word in the sentence
        w = tokenize(pattern)
        # add to our words list
        all_words.extend(w)
        # add to xy pair
        xy.append((w, tag))

In [ ]:
ignore_words = ['?', '.', '!']
all_words = [stem(w) for w in all_words if w not in ignore_words]
# remove duplicates and sort
all_words = sorted(set(all_words))
tags = sorted(set(tags))

In [ ]:
print(len(xy), "patterns")
print(len(tags), "tags:", tags)
print(len(all_words), "unique stemmed words:", all_words)

42 patterns
9 tags: ['confirm_order', 'delivery_time', 'goodbye', 'greeting', 'menu_recommendation', 'order', 'payments', 'show_menu', 'thanks']
71 unique stemmed words: ["'d", "'m", "'s", 'a', 'accept', 'all', 'an', 'ani', 'anyon', 'are', 'arriv', 'bye', 'can', 'card', 'credit', 'day', 'deliveri', 'do', 'doe', 'done', 'finish', 'get', 'good', 'goodby', 'have', 'hello', 'help', 'hey', 'hi', 'how', 'i', 'is', 'it', 'later', 'like', 'long', 'lot', 'me', 'menu', 'method', 'much', 'my', 'on', 'order', 'pay', 'payment', 'paypal', 'place', 'popular', 'recommend', 's', 'see', 'show', 'specialti', 'take', 'talk', 'thank', 'that', 'the', 'there', 'to', 'use', 'veri', 'want', 'what', 'when', 'will', 'with', 'you', 'your', '’']


In [ ]:
# create training data
X_train = []
y_train = []
for (pattern_sentence, tag) in xy:
    # X: bag of words for each pattern_sentence
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    # y: PyTorch CrossEntropyLoss needs only class labels, not one-hot
    label = tags.index(tag)
    y_train.append(label)

In [ ]:
X_train = np.array(X_train)
y_train = np.array(y_train)

In [ ]:
# Hyper-parameters
num_epochs = 1000
batch_size = 8
learning_rate = 0.001
input_size = len(X_train[0])
hidden_size = 8
output_size = len(tags)
print(input_size, output_size)


71 9


In [ ]:
class ChatDataset(Dataset):

    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = X_train
        self.y_data = y_train

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [ ]:
dataset = ChatDataset()
train_loader = DataLoader(dataset=dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = NeuralNet(input_size, hidden_size, output_size).to(device)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Train the model
for epoch in range(num_epochs):
    for (words, labels) in train_loader:
        words = words.to(device)
        labels = labels.to(dtype=torch.long).to(device)

        # Forward pass
        outputs = model(words)
       
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        print (f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [100/1000], Loss: 0.4407
Epoch [200/1000], Loss: 0.1081
Epoch [300/1000], Loss: 0.0206
Epoch [400/1000], Loss: 0.0036
Epoch [500/1000], Loss: 0.0014
Epoch [600/1000], Loss: 0.0026
Epoch [700/1000], Loss: 0.0014
Epoch [800/1000], Loss: 0.0005
Epoch [900/1000], Loss: 0.0002
Epoch [1000/1000], Loss: 0.0003


In [ ]:
print(f'final loss: {loss.item():.4f}')

final loss: 0.0003


In [ ]:
data = {
"model_state": model.state_dict(),
"input_size": input_size,
"hidden_size": hidden_size,
"output_size": output_size,
"all_words": all_words,
"tags": tags
}


In [ ]:
FILE = "data.pth"
torch.save(data, FILE)

print(f'training complete. file saved to {FILE}')

training complete. file saved to data.pth


# chat

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
with open('intents.json', 'r') as json_data:
    intents = json.load(json_data)

In [ ]:
FILE = "data.pth"
data = torch.load(FILE)

In [ ]:
input_size = data["input_size"]
hidden_size = data["hidden_size"]
output_size = data["output_size"]
all_words = data['all_words']
tags = data['tags']
model_state = data["model_state"]

In [ ]:
model = NeuralNet(input_size, hidden_size, output_size).to(device)
model.load_state_dict(model_state)
model.eval()

NeuralNet(
  (l1): Linear(in_features=71, out_features=8, bias=True)
  (l2): Linear(in_features=8, out_features=8, bias=True)
  (l3): Linear(in_features=8, out_features=9, bias=True)
  (relu): ReLU()
)

In [ ]:
bot_name = "Sam"
print("Let's chat! (type 'quit' to exit)")
while True:
    # sentence = "do you use credit cards?"
    sentence = input("You: ")
    if sentence == "quit":
        break

    sentence = tokenize(sentence)
    X = bag_of_words(sentence, all_words)
    X = X.reshape(1, X.shape[0])
    X = torch.from_numpy(X).to(device)

    output = model(X)
    _, predicted = torch.max(output, dim=1)

    tag = tags[predicted.item()]

    probs = torch.softmax(output, dim=1)
    prob = probs[0][predicted.item()]
    if prob.item() > 0.75:
        for intent in intents['intents']:
            if tag == intent["tag"]:
                print(f"{bot_name}: {random.choice(intent['responses'])}")
    else:
        print(f"{bot_name}: I do not understand...")

Let's chat! (type 'quit' to exit)
You: hi
Sam: Hi there! How can I assist you today?
You: i need shusi
Sam: I do not understand...
You: i want to order shusi
Sam: Sure! Please tell me what you'd like from our menu.
You: tell me the menu
Sam: Here's our menu: 
1. Spicy Tuna Roll
2. California Roll
3. Salmon Nigiri
4. Eel Avocado Roll
5. Shrimp Tempura Roll
6. Miso Soup
7. Edamame
8. Green Tea Ice Cream
9. Sushi Combo Platter
You: i want to have tuna roll
Sam: Great! What would you like to order?
You: tuna roll
Sam: I do not understand...
You: that's it
Sam: Thank you for your order! Do you need anything else?
You: no
Sam: I do not understand...
You: quit
